
# 練習問題: 二重ループを collapse(2) で並列化する

## 目標

`collapse(2)` 節を使い, **二重ループ** (2次元グリッド) をまとめて複数スレッドに分担させられることを体験する. 外側ループの繰り返し回数がスレッド数に比べて少ないとき, collapse がなぜ役立つのかを理解する.

## 課題

`grid2d.cpp` (または `grid2d.f90`) は, 2次元配列 `a[i][j]` (`N = 4`) に `i*10 + j` を書き込む二重ループを, 現状では1スレッドで実行している. 各 `(i, j)` をどのスレッドが処理したかも表示する.

コメント `TODO` の指示に従って **OpenMP の指示行を追加** し, 二重ループ全体を複数スレッドに分担させよ.

- C++: 外側 `for` ループの直前に `#pragma omp parallel for collapse(2)` を1行加える.
- Fortran: 二重 `do` ループを `!$omp parallel do collapse(2)` と `!$omp end parallel do` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore grid2d.cpp -o grid2d.exe

# Fortran
nvfortran -fast -mp=multicore grid2d.f90 -o grid2d.exe
```

```
OMP_NUM_THREADS=4 ./grid2d.exe
```

## 期待される結果

全 `N*N = 16` 個の `(i, j)` が, 複数のスレッドに分担される (順番は実行ごとに入れ替わってよい). 例:

```
a(0,0) = 0.000000  (thread 0)
a(0,1) = 1.000000  (thread 0)
a(2,0) = 20.000000  (thread 2)
a(1,1) = 11.000000  (thread 1)
...
```

## さらに考える

`collapse(2)` を外して (外側ループだけを並列化して) 実行し, スレッド番号の現れ方を比べよ.

- collapse なしでは, 外側ループの `i` (4回ぶん) だけがスレッドに配られる. 外側が4回しかないと, スレッドが4個より多くても5個目以降のスレッドは仕事をもらえない.
- `collapse(2)` を付けると `i` と `j` を合わせた `4*4 = 16` 回ぶんが配られ, より多くのスレッドに均等に分担できる.

外側ループの繰り返し回数がスレッド数に比べて少ないときほど, collapse の効果が大きいことを確認せよ.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:grid2d.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:grid2d.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ grid2d.cpp
#include <cstdio>
#include <omp.h>

int main() {
  const int N = 4;
  double a[N][N];
  // TODO: 下の二重ループの直前に #pragma omp parallel for collapse(2) を1行追加し, 二重ループ全体を複数スレッドに分担させよ.
  for (int i = 0; i < N; i++) {
    for (int j = 0; j < N; j++) {
      a[i][j] = i * 10 + j;
      printf("a[%d][%d] = %g  (thread %d)\n",
             i, j, a[i][j], omp_get_thread_num());
    }
  }
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore grid2d.cpp -o grid2d_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./grid2d_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./grid2d_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./grid2d_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:grid2d.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ grid2d.f90
program grid2d
  use omp_lib
  integer, parameter :: N = 4
  real(8) :: a(0:N-1, 0:N-1)
  integer :: i, j
  ! TODO: 下の二重ループを !$omp parallel do collapse(2) で始め, 二重ループ全体を複数スレッドに分担させよ.
  do i = 0, N - 1
     do j = 0, N - 1
        a(i, j) = i * 10 + j
        print "(a,i0,a,i0,a,f0.6,a,i0,a)", &
             "a(", i, ",", j, ") = ", a(i, j), "  (thread ", omp_get_thread_num(), ")"
     end do
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
end program grid2d

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore grid2d.f90 -o grid2d_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./grid2d_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./grid2d_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./grid2d_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:grid2d.f90}